# EPIC — audit пяти конкурсных assemblies перед переносом EXP047-B

Этот notebook **ничего не обучает** и не читает test labels.

Он проверяет пять конкурсных видов, обнаруживает реальные имена файлов,
валидирует whitelist/template, train/test contig separation и sparse csRNA
replicate BEDs. Результат — точная карта файлов, на которой затем можно
безопасно сделать species-agnostic data contract.


In [1]:
from pathlib import Path
import hashlib, json, os, subprocess, sys
import pandas as pd
from IPython.display import display

EXPECTED_PYTHON = Path(r"C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe").resolve()
assert Path(sys.executable).resolve() == EXPECTED_PYTHON, (sys.executable, EXPECTED_PYTHON)

CURRENT = Path.cwd().resolve()
PROJECT_ROOT = next(
    c for c in (CURRENT, *CURRENT.parents)
    if (c / "README.md").is_file()
    and (c / "src/ml_project").is_dir()
    and (c / "data/raw/unpacked").is_dir()
)

AGENTS = next(
    (p for p in (PROJECT_ROOT/"AGENTS.md", PROJECT_ROOT.parent/"AGENTS.md") if p.is_file()),
    None,
)
assert AGENTS is not None, "AGENTS.md not found"
print("AGENTS.md:", AGENTS)
print(AGENTS.read_text(encoding="utf-8"))

SCRIPT = PROJECT_ROOT / "scripts/audit_epic_5species_inputs.py"
SOURCE = 'from __future__ import annotations\n\nimport argparse\nimport gzip\nimport json\nfrom pathlib import Path\nimport sys\nfrom typing import Iterable\n\nimport numpy as np\nimport pandas as pd\n\n\nASSEMBLIES = [\n    "ilPloInte3.2",\n    "cgigas_uk_roslin_v1",\n    "Ofas_2.0",\n    "ASM119413v2",\n    "sSquSuc5.hap2",\n]\n\n\ndef project_root(start: Path | str | None = None) -> Path:\n    current = Path(start or Path.cwd()).resolve()\n    for candidate in (current, *current.parents):\n        if (\n            (candidate / "README.md").is_file()\n            and (candidate / "src/ml_project").is_dir()\n            and (candidate / "data/raw/unpacked").is_dir()\n        ):\n            return candidate\n    raise FileNotFoundError(f"EPIC project root not found above {current}")\n\n\ndef read_bed3(path: Path) -> pd.DataFrame:\n    return pd.read_csv(\n        path,\n        sep="\\t",\n        header=None,\n        usecols=[0, 1, 2],\n        names=["contig", "start", "end"],\n        dtype={"contig": str, "start": "int64", "end": "int64"},\n        compression="infer",\n    )\n\n\ndef read_bed6(path: Path) -> pd.DataFrame:\n    return pd.read_csv(\n        path,\n        sep="\\t",\n        header=None,\n        names=["contig", "start", "end", "name", "score", "strand"],\n        dtype={"contig": str, "start": "int64", "end": "int64", "strand": str},\n        compression="infer",\n    )\n\n\ndef validate_whitelist(frame: pd.DataFrame, contig_lengths: dict[str, int], label: str) -> None:\n    if frame.empty:\n        raise ValueError(f"{label}: empty whitelist")\n    if not set(frame["contig"]).issubset(contig_lengths):\n        unknown = sorted(set(frame["contig"]) - set(contig_lengths))[:10]\n        raise ValueError(f"{label}: unknown contigs {unknown}")\n    if (frame["start"] < 0).any() or (frame["end"] <= frame["start"]).any():\n        raise ValueError(f"{label}: invalid BED bounds")\n    max_len = frame["contig"].map(contig_lengths)\n    if (frame["end"] > max_len).any():\n        raise ValueError(f"{label}: interval outside contig")\n    ordered = frame.sort_values(["contig", "start", "end"], kind="stable")\n    for contig, group in ordered.groupby("contig", sort=False):\n        starts = group["start"].to_numpy(np.int64)\n        ends = group["end"].to_numpy(np.int64)\n        if len(group) > 1 and np.any(starts[1:] < ends[:-1]):\n            raise ValueError(f"{label}: overlapping intervals on {contig}")\n\n\ndef validate_template(template: pd.DataFrame, whitelist: pd.DataFrame, label: str) -> dict:\n    if template.empty:\n        raise ValueError(f"{label}: empty template")\n    if not set(template["strand"]) == {"+", "-"}:\n        raise ValueError(f"{label}: expected both strands")\n    expected = pd.concat(\n        [\n            whitelist.assign(strand="+"),\n            whitelist.assign(strand="-"),\n        ],\n        ignore_index=True,\n    )\n    cols = ["contig", "start", "end", "strand"]\n    left = template[cols].reset_index(drop=True)\n    right = expected[cols].reset_index(drop=True)\n    pd.testing.assert_frame_equal(left, right, check_dtype=False)\n    lengths = template["end"].to_numpy(np.int64) - template["start"].to_numpy(np.int64)\n    return {\n        "template_rows": int(len(template)),\n        "position_strand_count": int(lengths.sum()),\n    }\n\n\ndef load_fai(path: Path) -> dict[str, int]:\n    fai = pd.read_csv(\n        path,\n        sep="\\t",\n        header=None,\n        names=["contig", "length", "offset", "line_bases", "line_bytes"],\n    )\n    if fai.empty or not fai["contig"].is_unique:\n        raise ValueError(f"Bad FAI: {path}")\n    return dict(zip(fai["contig"].astype(str), fai["length"].astype("int64")))\n\n\ndef choose_single(candidates: list[Path], label: str, required: bool = True) -> Path | None:\n    candidates = sorted(set(candidates))\n    if len(candidates) == 1:\n        return candidates[0]\n    if not candidates and not required:\n        return None\n    raise RuntimeError(\n        f"{label}: expected exactly one file, found {len(candidates)}: "\n        + ", ".join(str(x.name) for x in candidates)\n    )\n\n\ndef identify_files(root: Path, assembly: str) -> dict[str, Path | None | list[Path]]:\n    source = root / "data/raw/unpacked" / assembly\n    participants = source / "for_participants"\n    genome = source / "genome"\n    if not participants.is_dir():\n        raise FileNotFoundError(participants)\n    if not genome.is_dir():\n        raise FileNotFoundError(genome)\n\n    fasta_candidates = [\n        p for p in [genome / "genome.fa.gz", genome / "genome.fa"]\n        if p.is_file()\n    ]\n    fasta = choose_single(fasta_candidates, f"{assembly} FASTA")\n    fai = choose_single(\n        [p for p in [genome / "genome.fa.fai", genome / "genome.fa.gz.fai"] if p.is_file()],\n        f"{assembly} FAI",\n    )\n\n    whitelist_train = choose_single(\n        list(participants.glob("whitelist.train.bed*")),\n        f"{assembly} whitelist.train",\n    )\n    whitelist_test = choose_single(\n        list(participants.glob("whitelist.test.bed*")),\n        f"{assembly} whitelist.test",\n    )\n    template_train = choose_single(\n        list(participants.glob("template.train.bed*")),\n        f"{assembly} template.train BED",\n    )\n    template_test = choose_single(\n        list(participants.glob("template.test.bed*")),\n        f"{assembly} template.test BED",\n    )\n\n    # Filtered sparse csRNA replicate BEDs only; explicitly exclude *unfiltered*.\n    r1 = sorted(\n        p for p in participants.glob("csRNA*-r1-train.bed*")\n        if "unfiltered" not in p.name.lower()\n    )\n    r2 = sorted(\n        p for p in participants.glob("csRNA*-r2-train.bed*")\n        if "unfiltered" not in p.name.lower()\n    )\n\n    # Match r1/r2 by replacing the literal replicate token.\n    pairs = []\n    r2_by_key = {\n        p.name.replace("-r2-train", "-rX-train"): p for p in r2\n    }\n    for p1 in r1:\n        key = p1.name.replace("-r1-train", "-rX-train")\n        p2 = r2_by_key.get(key)\n        if p2 is not None:\n            pairs.append((p1, p2))\n\n    dense_train = sorted(participants.glob("csRNA-train.txt*"))\n    dense_train_path = choose_single(\n        dense_train,\n        f"{assembly} pooled csRNA-train.txt",\n        required=False,\n    )\n\n    return {\n        "source": source,\n        "participants": participants,\n        "fasta": fasta,\n        "fai": fai,\n        "whitelist_train": whitelist_train,\n        "whitelist_test": whitelist_test,\n        "template_train": template_train,\n        "template_test": template_test,\n        "r1_candidates": r1,\n        "r2_candidates": r2,\n        "replicate_pairs": pairs,\n        "dense_train": dense_train_path,\n    }\n\n\ndef interval_contains(\n    whitelist_lookup: dict[str, tuple[np.ndarray, np.ndarray]],\n    contig: str,\n    start: int,\n    end: int,\n) -> bool:\n    item = whitelist_lookup.get(contig)\n    if item is None:\n        return False\n    starts, ends = item\n    idx = int(np.searchsorted(starts, start, side="right") - 1)\n    return idx >= 0 and start >= int(starts[idx]) and end <= int(ends[idx])\n\n\ndef sparse_bed_stats(path: Path, whitelist: pd.DataFrame) -> dict:\n    ordered = whitelist.sort_values(["contig", "start"], kind="stable")\n    lookup = {\n        str(contig): (\n            group["start"].to_numpy(np.int64),\n            group["end"].to_numpy(np.int64),\n        )\n        for contig, group in ordered.groupby("contig", sort=False)\n    }\n    rows = 0\n    positive_position_strands = 0\n    total_count_mass = 0\n    opener = gzip.open if path.suffix.lower() == ".gz" else open\n    with opener(path, "rt", encoding="ascii") as stream:\n        for line_no, line in enumerate(stream, start=1):\n            fields = line.split()\n            if len(fields) != 6:\n                raise ValueError(f"{path.name}:{line_no}: expected BED6")\n            contig, start, end, _name, count, strand = fields\n            start, end, count = int(start), int(end), int(count)\n            if end <= start or count <= 0 or strand not in {"+", "-"}:\n                raise ValueError(f"{path.name}:{line_no}: invalid signal row")\n            if not interval_contains(lookup, contig, start, end):\n                raise ValueError(\n                    f"{path.name}:{line_no}: signal interval outside train whitelist"\n                )\n            width = end - start\n            rows += 1\n            positive_position_strands += width\n            total_count_mass += width * count\n    return {\n        "rows": rows,\n        "positive_position_strands": positive_position_strands,\n        "total_count_mass": total_count_mass,\n    }\n\n\ndef rel(root: Path, path: Path | None) -> str | None:\n    return None if path is None else path.relative_to(root).as_posix()\n\n\ndef audit_assembly(root: Path, assembly: str) -> tuple[dict, dict]:\n    files = identify_files(root, assembly)\n    contig_lengths = load_fai(files["fai"])\n\n    w_train = read_bed3(files["whitelist_train"])\n    w_test = read_bed3(files["whitelist_test"])\n    validate_whitelist(w_train, contig_lengths, f"{assembly} train whitelist")\n    validate_whitelist(w_test, contig_lengths, f"{assembly} test whitelist")\n\n    train_contigs = set(w_train["contig"].astype(str))\n    test_contigs = set(w_test["contig"].astype(str))\n    overlap = sorted(train_contigs & test_contigs)\n    if overlap:\n        raise ValueError(f"{assembly}: official train/test contigs overlap: {overlap[:10]}")\n\n    t_train = read_bed6(files["template_train"])\n    t_test = read_bed6(files["template_test"])\n    train_contract = validate_template(t_train, w_train, f"{assembly} template.train")\n    test_contract = validate_template(t_test, w_test, f"{assembly} template.test")\n\n    pairs = files["replicate_pairs"]\n    if len(pairs) != 1:\n        raise RuntimeError(\n            f"{assembly}: expected one filtered csRNA r1/r2 pair, found {len(pairs)}: "\n            + ", ".join(f"{a.name} + {b.name}" for a, b in pairs)\n        )\n    r1, r2 = pairs[0]\n    r1_stats = sparse_bed_stats(r1, w_train)\n    r2_stats = sparse_bed_stats(r2, w_train)\n\n    expected_current_r1 = files["participants"] / "csRNA-r1-train.bed.gz"\n    expected_current_r2 = files["participants"] / "csRNA-r2-train.bed.gz"\n    current_epic_data_compatible = expected_current_r1.is_file() and expected_current_r2.is_file()\n\n    row = {\n        "assembly": assembly,\n        "fasta": rel(root, files["fasta"]),\n        "fai": rel(root, files["fai"]),\n        "train_contigs": len(train_contigs),\n        "test_contigs": len(test_contigs),\n        "train_whitelist_intervals": len(w_train),\n        "test_whitelist_intervals": len(w_test),\n        "train_position_strands": train_contract["position_strand_count"],\n        "test_position_strands": test_contract["position_strand_count"],\n        "r1_file": rel(root, r1),\n        "r2_file": rel(root, r2),\n        "r1_sparse_positive_positions": r1_stats["positive_position_strands"],\n        "r2_sparse_positive_positions": r2_stats["positive_position_strands"],\n        "r1_total_count_mass": r1_stats["total_count_mass"],\n        "r2_total_count_mass": r2_stats["total_count_mass"],\n        "pooled_dense_train_file": rel(root, files["dense_train"]),\n        "current_epic_data_hardcoded_paths_work": current_epic_data_compatible,\n        "status": "PASS",\n    }\n\n    mapping = {\n        "assembly": assembly,\n        "source_dir": rel(root, files["source"]),\n        "genome": {\n            "fasta": rel(root, files["fasta"]),\n            "fai": rel(root, files["fai"]),\n        },\n        "participants": {\n            "whitelist_train": rel(root, files["whitelist_train"]),\n            "whitelist_test": rel(root, files["whitelist_test"]),\n            "template_train": rel(root, files["template_train"]),\n            "template_test": rel(root, files["template_test"]),\n            "csrna_r1_filtered": rel(root, r1),\n            "csrna_r2_filtered": rel(root, r2),\n            "csrna_dense_pooled": rel(root, files["dense_train"]),\n        },\n        "contract": {\n            "official_train_test_contigs_disjoint": True,\n            "template_train_exactly_whitelist_plus_then_minus": True,\n            "template_test_exactly_whitelist_plus_then_minus": True,\n            "sparse_replicates_inside_train_whitelist": True,\n            "train_position_strands": train_contract["position_strand_count"],\n            "test_position_strands": test_contract["position_strand_count"],\n        },\n    }\n    return row, mapping\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--project-root", default=None)\n    parser.add_argument("--output-dir", default=None)\n    args = parser.parse_args()\n\n    root = project_root(args.project_root)\n    output = (\n        Path(args.output_dir).resolve()\n        if args.output_dir\n        else root / "artifacts" / "audits" / "EPIC_5SPECIES_INPUT_AUDIT"\n    )\n    output.mkdir(parents=True, exist_ok=True)\n\n    rows = []\n    mappings = {}\n    failures = []\n    for assembly in ASSEMBLIES:\n        print(f"\\n=== {assembly} ===", flush=True)\n        try:\n            row, mapping = audit_assembly(root, assembly)\n            rows.append(row)\n            mappings[assembly] = mapping\n            print("PASS", flush=True)\n            print(\n                f"train={row[\'train_position_strands\']:,} "\n                f"test={row[\'test_position_strands\']:,} "\n                f"r1={row[\'r1_file\']} r2={row[\'r2_file\']}",\n                flush=True,\n            )\n        except Exception as error:\n            failures.append({"assembly": assembly, "error": repr(error)})\n            rows.append({"assembly": assembly, "status": "FAIL", "error": repr(error)})\n            print(f"FAIL: {error!r}", flush=True)\n\n    frame = pd.DataFrame(rows)\n    frame.to_csv(output / "assembly_audit.csv", index=False)\n    (output / "assembly_file_map.json").write_text(\n        json.dumps(mappings, ensure_ascii=False, indent=2) + "\\n",\n        encoding="utf-8",\n    )\n    (output / "audit_failures.json").write_text(\n        json.dumps(failures, ensure_ascii=False, indent=2) + "\\n",\n        encoding="utf-8",\n    )\n\n    print("\\n=== SUMMARY ===")\n    print(frame.to_string(index=False))\n    print(f"\\nArtifacts: {output}")\n    if failures:\n        raise SystemExit(f"Audit failed for {len(failures)} assemblies; see audit_failures.json")\n\n\nif __name__ == "__main__":\n    main()\n'
EXPECTED_SHA = "a57d9dc5bdd95a313247a57751e2515f5653a7e537ec7715597146c59f18ad1a"
assert hashlib.sha256(SOURCE.encode("utf-8")).hexdigest() == EXPECTED_SHA
compile(SOURCE, str(SCRIPT), "exec")

if SCRIPT.exists():
    existing = hashlib.sha256(SCRIPT.read_bytes()).hexdigest()
    assert existing == EXPECTED_SHA, (
        f"Existing audit script differs: {existing} != {EXPECTED_SHA}"
    )
else:
    SCRIPT.write_text(SOURCE, encoding="utf-8")

OUTPUT = PROJECT_ROOT / "artifacts/audits/EPIC_5SPECIES_INPUT_AUDIT"
print("Project:", PROJECT_ROOT)
print("Audit script:", SCRIPT)
print("SHA256:", EXPECTED_SHA)


AGENTS.md: D:\Projects\EPIC\AGENTS.md
# Project agent instructions

- Run all Python commands, notebooks, analyses, and tests for this workspace in the Conda environment `epic-eda`.
- On this Windows machine, prefer the explicit interpreter `C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe` when invoking Python non-interactively.
- Do not use the repository `.venv` or the system Python for project work unless the user explicitly requests it.

## Decision-making and scientific judgment

- Do not automatically agree with the user's proposed interpretation, experiment, architecture, or next step.
- Treat the user's preferred direction as a hypothesis to evaluate, not as the conclusion to defend.
- Independently compare the available alternatives using experimental evidence, expected information gain, methodological validity, cost, risk, and relevance to the primary objective.
- Recommend the path that is most logically supported by the evidence, even when it conflicts with the use

## Запустить audit

Важный момент: для csRNA берутся только **filtered** `r1/r2` BED-файлы.
Файлы с `unfiltered` исключаются. Dense `csRNA-train.txt.gz` только
обнаруживается; полностью читать его на этом шаге не нужно.


In [5]:
cmd = [str(EXPECTED_PYTHON), "-u", str(SCRIPT), "--project-root", str(PROJECT_ROOT)]
print("RUN:", subprocess.list2cmdline(cmd), flush=True)
p = subprocess.run(cmd, cwd=PROJECT_ROOT)
if p.returncode != 0:
    raise RuntimeError(f"Audit failed with code {p.returncode}; inspect audit_failures.json")


RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\audit_epic_5species_inputs.py --project-root D:\Projects\EPIC\EPIC


## Результат


In [6]:
from pathlib import Path

path = PROJECT_ROOT / "scripts/audit_epic_5species_inputs.py"

text = path.read_text(encoding="utf-8")

old = """            start, end, count = int(start), int(end), int(count)
            if end <= start or count <= 0 or strand not in {"+", "-"}:
"""

new = """            start, end = int(start), int(end)
            count_value = float(count)

            if not count_value.is_integer():
                raise ValueError(
                    f"{path.name}:{line_no}: non-integer csRNA count {count!r}"
                )

            count = int(count_value)

            if end <= start or count <= 0 or strand not in {"+", "-"}:
"""

assert old in text, "Expected audit code was not found"

path.write_text(text.replace(old, new), encoding="utf-8")

print("PATCHED:", path)

AssertionError: Expected audit code was not found

In [3]:
print((OUTPUT / "audit_failures.json").read_text(encoding="utf-8"))

[
  {
    "assembly": "sSquSuc5.hap2",
    "error": "ValueError(\"invalid literal for int() with base 10: '1.43057e+06'\")"
  }
]



In [ ]:
audit = pd.read_csv(OUTPUT/"assembly_audit.csv")
display(audit)

file_map = json.loads((OUTPUT/"assembly_file_map.json").read_text(encoding="utf-8"))
print(json.dumps(file_map, ensure_ascii=False, indent=2))
